# Investment strategy comparison dashboard

The end output is a set of charts comparing immediate investment with representative limit-order strategies on the same contribution-neutral basis.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

from retail_sp500.daily_data import daily_data_summary, load_or_fetch_twelve_data_daily

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

from retail_sp500.limit_plotting import (
    strategy_calmar_ranking_figure,
    strategy_drawdown_figure,
    strategy_return_drawdown_figure,
    strategy_wealth_figure,
)
from retail_sp500.limit_portfolio import (
    RecurringLimitConfig,
    compare_recurring_limit_strategies,
    evaluate_recurring_limit_grid,
    walk_forward_recurring_limit_selection,
)

In [ ]:
SYMBOL = "SPY"
START_DATE = "2007-06-01"
CACHE_PATH = Path("data/processed/spy_daily_1day.csv")
REFRESH = False

daily = load_or_fetch_twelve_data_daily(
    os.getenv("TWELVE_DATA_API_KEY"),
    cache_path=CACHE_PATH,
    refresh=REFRESH,
    symbol=SYMBOL,
    start_date=START_DATE,
)

source = daily_data_summary(daily, symbol=SYMBOL)
print(source)
assert source["interval"] == "1day"
assert daily.index.max() <= pd.Timestamp.today().normalize()
daily.tail()

## Derive representative strategies

In [ ]:
DISCOUNTS = np.arange(0.0, 0.0301, 0.001)
WAIT_HORIZONS = [1, 3, 5, 10, 20]

strategy_grid = pd.concat(
    [
        evaluate_recurring_limit_grid(
            daily,
            DISCOUNTS,
            max_wait_sessions=wait,
            initial_cash=100_000,
            monthly_contribution=1_000,
        ).assign(wait_horizon=wait)
        for wait in WAIT_HORIZONS
    ],
    ignore_index=True,
)

best_terminal = strategy_grid.loc[strategy_grid["ending_excess_value"].idxmax()]
best_calmar = strategy_grid.dropna(subset=["calmar_ratio"]).loc[
    strategy_grid.dropna(subset=["calmar_ratio"])["calmar_ratio"].idxmax()
]
walk_forward = walk_forward_recurring_limit_selection(
    daily,
    DISCOUNTS,
    train_years=5,
    test_years=1,
    max_wait_sessions=5,
    monthly_contribution=1_000,
    selection_metric="calmar_ratio",
)
walk_forward_discount = float(walk_forward["selected_discount"].median())

raw_strategies = [
    ("Previous-close limit", 0.0, 1),
    ("0.5% pullback", 0.005, 5),
    ("1.0% pullback", 0.010, 5),
    ("Best terminal value", float(best_terminal["discount"]), int(best_terminal["wait_horizon"])),
    ("Best full-sample Calmar", float(best_calmar["discount"]), int(best_calmar["wait_horizon"])),
    ("Walk-forward median", walk_forward_discount, 5),
]

strategies = {}
seen = set()
for label, discount, wait in raw_strategies:
    key = (round(discount, 10), wait)
    if key in seen:
        continue
    seen.add(key)
    strategies[f"{label}: {discount:.1%} below, {wait} session{'s' if wait != 1 else ''}"] = (
        RecurringLimitConfig(
            discount=discount,
            max_wait_sessions=wait,
            initial_cash=100_000,
            monthly_contribution=1_000,
        )
    )

strategy_metrics, strategy_curves = compare_recurring_limit_strategies(daily, strategies)
strategy_metrics.sort_values("calmar_ratio", ascending=False)

## Final strategy charts

In [ ]:
strategy_calmar_ranking_figure(strategy_metrics).show()

In [ ]:
strategy_return_drawdown_figure(strategy_metrics).show()

In [ ]:
strategy_wealth_figure(strategy_curves).show()

In [ ]:
strategy_drawdown_figure(strategy_curves).show()

## Reading the dashboard

- The Calmar ranking rewards annualized return per unit of worst drawdown.
- The return/drawdown scatter shows the trade-off directly.
- The wealth chart shows compounded contribution-neutral performance.
- The drawdown chart shows when each strategy suffered and recovered.

Full-sample winners are explicitly labelled in-sample. The walk-forward median is the more defensible candidate, but it is still not a live trading instruction.